<div align="center">
  <img src="assets/fleetguard-logo.svg" alt="FleetGuard AI" width="360">
</div>

# 02 · Healthy signal behaviour
**Goal:** understand every observed signal before deciding what a model should receive.

## Review at a glance

This is the **training healthy-control baseline** for the reviewed seed-42,
10-second dataset. No saved cell contains an execution error. Findings describe
this snapshot and must be refreshed if the source or configuration changes.

| Baseline check | Observed result |
| --- | --- |
| Healthy-control wagons | 3 |
| Received wagon / bogie / axle / wheel rows | 6,324 / 12,648 / 25,296 / 50,592 |
| Numerical field placements checked | 28; no missing or invalid sensor readings |
| Healthy constants | Available generators = 4; reset and communication-error counters = 0 |
| Typical context to preserve | Motion, brake demand, ambient temperature, load and component identity |

These are sample-level descriptions, not independent failure trials or certified
operating limits. Test signal values remain unopened.


In [1]:
import os
import sys
from pathlib import Path

# Works when Jupyter starts from the repository root or from nbs.
HELPERS = Path.cwd() if (Path.cwd() / "fleetguard_eda.py").exists() else Path.cwd() / "nbs"
sys.path.insert(0, str(HELPERS))
import fleetguard_eda as eda
import panel as pn
import plotly.express as px

eda.setup()
ROOT = eda.project_root()
SOURCE = Path(os.environ.get("FLEETGUARD_EDA_SOURCE", ROOT / "data/generated/portfolio-v1"))
FEATURES = Path(os.environ.get("FLEETGUARD_EDA_FEATURES", ROOT / "data/features/portfolio-v1"))
SPLIT = "train"  # change to validation to check findings; test is blocked for exploration
ws = eda.Workspace(SOURCE, FEATURES)
pn.pane.Markdown(
    f"**Source:** `{SOURCE}`  ·  **Features:** `{FEATURES}`  ·  **Explore:** `{SPLIT}`\n\n"
    "Maps need internet. Linked controls need a running, trusted notebook kernel."
)

Markdown(str)

### 1 · Signal availability and spread
Primary reference: healthy-control wagons only. Constants are visible, not discarded silently.

In [2]:
healthy_profile = eda.quality_profile(ws, SPLIT, variants=["healthy"])
eda.table(healthy_profile)

Tabulator(disabled=True, page_size=10, pagination='local', show_index=False, sizing_mode='stretch_width', value=    Level     ...)

**Availability:** all 28 numerical field placements have valid readings and no
missing/invalid sensor values in these healthy controls. Received rows reconcile
as **6,324 wagon → 12,648 bogie → 25,296 axle → 50,592 wheel** observations.
This covers three training healthy-control wagons, not all 48 fleet assets.

### 2 · Every numerical signal
Choose a component level and distribution view. **Density / exact values** uses KDE
for continuous signals, exact sample shares for low-cardinality values and an ECDF
when a large repeated value would be misleadingly smoothed. Histogram remains available.
An ECDF shows the fraction of valid readings at or below each value.

KDE is a visual approximation, not a probability of failure. Curves are limited to
and renormalised over the observed range; up to 4,000 readings are sampled with a
fixed seed per signal. Each subplot has its own units and vertical scale.

### Known properties
Wheel diameter and handbrake position are metadata, not new measurements each reporting instant.

In [3]:
eda.static_properties(ws, SPLIT)

Tabs
    [0] Tabulator(disabled=True, page_size=10, pagination='local', show_index=False, sizing_mode='stretch_width', value=  schema_version  ...)
    [1] Tabulator(disabled=True, page_size=10, pagination='local', show_index=False, sizing_mode='stretch_width', value=      asset_id  ...)
    [2] Plotly(Figure, config={'displaylogo': False}, sizing_mode='stretch_width')

**Metadata:** wheel diameter and handbrake equipment describe the physical
configuration. Repeating them in observation rows does not create independent
measurements; wheel/axle comparisons must preserve the full parent identity.

In [4]:
gallery_level = pn.widgets.Select(name="Component level", options=list(eda.NUMERIC))
gallery_view = pn.widgets.Select(
    name="Distribution view", options=["Density / exact values", "Histogram", "ECDF"]
)
pn.Column(
    pn.Row(gallery_level, gallery_view),
    pn.bind(
        lambda level, view: eda.pane(eda.distribution_gallery(ws, level, SPLIT, view)),
        gallery_level,
        gallery_view,
    ),
)

Column
    [0] Row
        [0] Select(label='Component level', name='Component level', options=['wagon', 'bogie', ...], value='wagon')
        [1] Select(label='Distribution view', name='Distribution view', options=['Density / exact values',...], value='Density / exact values')
    [1] ParamFunction(function, _pane=Plotly, defer_load=False)

**Distribution finding:** speed, acceleration, pressure and power signals need
operating context; a single normal curve is not an assumption of this simulator.
Battery voltage is concentrated near 4.2 V, while generator availability and two
controller counters are constant. Use the ECDF to inspect repeated values without
inventing a smooth spread between them.

### 3 · States and categories
Shared states are shown **once at wagon level**. Bogie rail condition repeats the
parent wagon context; axles and wheels add no categorical measurement fields in
this contract. Their position, side and handbrake attributes belong in the metadata
summary above. Counts describe received samples, not independent journeys.

In [5]:
eda.state_gallery(ws, "wagon", SPLIT)

Plotly(Figure, config={'displaylogo': False}, sizing_mode='stretch_width')

**Observed states:** 4,266 moving, 1,950 stationary and 108 braking reports.
Brake demand is release in 6,216 reports and apply in 108. Controller health is
always healthy. Dry, wet and leaf-contaminated rails all occur in healthy data:
rail condition alone is not a fault label. The count of four available generators
means availability, not that they power the controller while stationary.

### 4 · Inspect a signal, wagon and component
Compare an ECDF, component spread and chronological behaviour. The distribution
default is now an ECDF, so wagon comparisons do not depend on histogram bin width.
Histogram remains available. Empty combinations are explicit.

In [6]:
explore_level = pn.widgets.Select(name="Explore level", options=list(eda.NUMERIC))
pn.Column(
    explore_level, pn.bind(lambda level: eda.healthy_explorer(ws, level, SPLIT), explore_level)
)

Column
    [0] Select(label='Explore level', name='Explore level', options=['wagon', 'bogie', ...], value='wagon')
    [1] ParamFunction(function, _pane=Column, defer_load=False)

**Context:** compare the same signal and component across operating states
before interpreting a wide distribution as abnormal. Pooled differences can come
from wagon baselines, route exposure and motion rather than a fault.

### 5 · Linked selection
Box/lasso a region in the relationship plot. The same rows are selected in the time plot. At most 12,000 rows are shown.

In [7]:
eda.linked_relationship(ws, "axle", SPLIT)

:Layout
   .Points.I  :DynamicMap   []
      :Overlay
         .Points.I     :Points   [wheel_speed_kph,vibration_rms_g]   (asset_id,component,event_time)
         .Points.II    :Points   [wheel_speed_kph,vibration_rms_g]   (asset_id,component,event_time)
         .Rectangles.I :Rectangles   [x0,y0,x1,y1]
         .Path.I       :Path   [x,y]
   .Points.II :DynamicMap   []
      :Overlay
         .Points.I     :Points   [event_time,vibration_rms_g]   (asset_id,component,wheel_speed_kph)
         .Points.II    :Points   [event_time,vibration_rms_g]   (asset_id,component,wheel_speed_kph)
         .Rectangles.I :Rectangles   [x0,y0,x1,y1]
         .Path.I       :Path   [x,y]

**Linked view:** select a speed/vibration region and inspect when those same
rows occur. This separates operating regimes from isolated points. The linked view
is sampled to at most 12,000 rows and is an exploration aid, not a fleet-wide rate.

### 6 · Correlation, with engineering context
Spearman correlations pool healthy operating states. Repeated/time-dependent samples are not independent evidence; do not interpret causally.

### Engineering relationships
Inspect pressure response by brake demand and bearing heat above ambient by speed/load.

In [8]:
eda.engineering_relationships(ws, SPLIT)

Tabs
    [0] Plotly(Figure, config={'displaylogo': False}, sizing_mode='stretch_width')
    [1] Plotly(Figure, config={'displaylogo': False}, sizing_mode='stretch_width')

**Engineering context:** compare pressures with apply/release demand and inspect
bearing temperature above ambient alongside speed and axle load. These are
simulated relationships; apparent agreement does not validate real brake response
or establish a diagnostic threshold.

In [9]:
corr_level = pn.widgets.Select(name="Correlation level", options=list(eda.NUMERIC))
clustered = pn.widgets.Checkbox(name="Cluster related signals", value=False)
pn.Column(
    pn.Row(corr_level, clustered),
    pn.bind(
        lambda level, cluster: eda.correlation_view(ws, level, SPLIT, cluster),
        corr_level,
        clustered,
    ),
)

Column
    [0] Row
        [0] Select(label='Correlation level', name='Correlation level', options=['wagon', 'bogie', ...], value='wagon')
        [1] Checkbox(label='Cluster related signals', name='Cluster related signals')
    [1] ParamFunction(function, _pane=Plotly, defer_load=False)

**Correlation:** Spearman measures monotonic association, not causal influence.
The matrix pools states and repeated component reports. Shared wagon fields and
time-progress variables can produce strong associations by construction; review
states and redundancy before choosing model inputs. Constants have no defined
correlation, so a blank entry is not evidence of independence.

### 7 · Geographic context for healthy controls
Report density reflects routes/dwell, not failure risk.

In [10]:
healthy_wagons = ws.load("wagon", SPLIT, variants=["healthy"])
eda.folium_map(healthy_wagons)

**Geography:** the three control wagons are a small exposure sample. Route/dwell
concentrations on the map are not evidence of reliability or local failure risk.
Keep the train/validation split boundary when investigating location patterns.

### 8 · Check simulator fingerprints
Inspect constants, exact wheelset/wagon speed relationships, repeated baselines and the power transition. These can be designed simulator relationships, not evidence of real hardware behaviour.

In [11]:
varying = healthy_profile.assign(Constant=healthy_profile.Distinct <= 1)
eda.pane(
    eda.figure(
        px.bar(varying, x="Signal", y="Distinct", color="Level", hover_data=["Unit", "Constant"]),
        "Distinct valid values — constants and low variation need review",
    )
)

Plotly(Figure, config={'displaylogo': False}, sizing_mode='stretch_width')

**Simulator fingerprints:** all healthy generator counts are 4; controller reset
and sensor-communication-error counts are 0. Battery voltage spans only
**4.1911–4.2000 V** on these duties. Do not discard these signals automatically:
some are intentionally expected to change during the fault scenarios in notebook 03.

### 9 · Decisions and next session

| Finding | Modelling implication |
| --- | --- |
| Healthy controls are complete for the measured numerical fields | Keep them as the baseline; this is not a claim that production telemetry will be complete. |
| States create different normal regimes | Compare like operating contexts; use state-aware features or evaluation where appropriate. |
| Shared wagon states repeat in component rows | Display once and avoid treating duplicates as independent evidence. |
| Speed/rotation follow generator rules | Check simulator shortcuts and redundant predictors in the anomaly session. |
| Several healthy fields are constant | Review their fault behaviour before deciding whether they belong in a model or a separate monitoring rule. |
| Battery is almost fully charged in these short duties | This dataset does not validate multi-day standby endurance or real battery ageing. |
| Units and scales differ substantially | Choose scaling/transforms using training data when modelling, not global test statistics. |

**Useful observed ranges, not alarm thresholds:** healthy bearing temperature
31.898–51.743 °C; axle vibration 0.1174–0.4387 g; brake-pipe pressure
3.345–5.137 bar; brake-cylinder pressure 0–2.904 bar. Interpret them with state,
ambient conditions and component context, not as standalone failure boundaries.

**Next: `03_anomaly_signal_behaviour.ipynb`.** Compare affected and unaffected
components, inspect raw 10-second reports versus minute summaries, and test
whether missingness, fixed offsets or timing make the synthetic faults too easy.
No model inputs, thresholds or model-performance claims are finalised here.
